# 第7回 埋め込み — 言葉の意味を数字で測る

**このノートブックは第7回だけで完結します。** プログラミングの経験は要りません。

やること:
1. 文を384個の数に変え、2つの文の「近さ」を測る
2. キーワード検索と意味検索を比べる
3. ラベルなしで話題ごとに分かれることを確かめる
4. 否定が見分けられないことを確かめる(今日いちばん重要)

所要時間:実習 約25分 / 課題 約40分

## 0. 準備

上から順にセルを実行してください(セルを選んで **Shift + Enter**)。
最初の2つのセルは、合わせて1〜2分かかります。

In [ ]:
# 準備(このセルを最初に1回だけ実行。1〜2分かかります)
!pip install -q sentence-transformers japanize-matplotlib
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt, japanize_matplotlib
from sentence_transformers import SentenceTransformer
plt.rcParams["font.size"] = 12
print("準備できました")

In [ ]:
# 埋め込みモデルを読み込む(初回は1分ほどダウンロードします)
# 日本語も英語も扱える、小さめのモデルです。
モデル = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

def 数字にする(文リスト):
    """文のリストを受け取り、1文あたり384個の数にして返す(長さ1に揃える)"""
    v = モデル.encode(文リスト)
    return v / np.linalg.norm(v, axis=1, keepdims=True)

v = 数字にする(["猫が好きです"])
print("1文あたりの数の個数:", v.shape[1])
print("最初の8個:", np.round(v[0][:8], 3))
print("\nこの384個の数が、この文の「意味」です。")
print("人間には何を表しているのか読み取れません。読み取れないまま使います。")

## 1. 2つの文の「近さ」を測る

**埋め込み**とは、文を決まった個数の数字に変える操作です。
そして2つの文の近さは、**同じ位置の数どうしを掛けて、全部足す**だけで求まります。

1 に近いほど「意味が近い」、0 に近いほど「無関係」です。

In [ ]:
# ===== ① 2つの文の「近さ」を測る =====
def 近さ(文A, 文B):
    v = 数字にする([文A, 文B])
    return float(v[0] @ v[1])        # 掛けて足すだけ

ペア = [
    ("パソコンが動かない", "PCが起動しません"),
    ("パソコンが動かない", "パソコンが壊れた"),
    ("パソコンが動かない", "今日は良い天気です"),
    ("猫が好きです", "ネコが好きです"),
    ("猫が好きです", "ねこがすきです"),
    ("会議を来週に延期したい", "ミーティングを翌週にずらしたい"),
]
print(f"{'近さ':>6}  文A  ×  文B")
print("-" * 62)
for a, b in sorted(ペア, key=lambda p: -近さ(*p)):
    print(f"{近さ(a, b):>6.3f}  「{a}」×「{b}」")

print()
print("読みかたの目安(このモデルの場合):")
print("  0.85以上=ほぼ同じ意味 / 0.6〜0.85=関連が強い / 0.3〜0.6=なんとなく / 0.3未満=ほぼ無関係")
print("※ この目安はモデルごとに違います。絶対値ではなく、同じモデルの中で相対的に比べてください。")

> **注目**:「パソコンが動かない」と「PCが起動しません」には、共通の文字がほとんどありません。
> それでも高い数字が出ます。辞書を引いているわけではありません。
>
> 一方で「猫が好きです」を全部ひらがなにすると、近さが大きく落ちます。
> 文の区切りかた(第1回のトークン)が変わってしまうためです。

## 2. 何が変わるのか — キーワード検索と比べる

素朴なキーワード検索(共通する文字の割合)と、意味検索を比べます。
社内の問い合わせ集から、「パソコンの電源が入らない」に合う文書を探させます。

In [ ]:
# ===== ② キーワード検索 vs 意味検索 =====
問い合わせ集 = [
    "PCが起動しません",
    "会議室の予約方法を教えてください",
    "経費精算の締切はいつですか",
    "有給休暇の申請手順",
    "社内Wi-Fiにつながらない",
    "給与明細はどこで見られますか",
]

def 文字の一致(質問, 文書):
    """素朴なキーワード検索:共通する文字の割合"""
    return len(set(質問) & set(文書)) / len(set(質問))

def 検索する(質問, 文書リスト):
    V = 数字にする(文書リスト)
    q = 数字にする([質問])[0]
    意味 = V @ q
    文字 = np.array([文字の一致(質問, d) for d in 文書リスト])
    print(f"質問:「{質問}」\n")
    print(f"{'文書':<26}{'文字の一致':>10}{'意味の近さ':>11}")
    print("-" * 50)
    for d, m, k in zip(文書リスト, 意味, 文字):
        print(f"{d:<26}{k:>10.2f}{m:>11.3f}")
    print()
    print(f"  文字一致の1位 → {文書リスト[int(np.argmax(文字))]}")
    print(f"  意味検索の1位 → {文書リスト[int(np.argmax(意味))]}")
    return 意味, 文字

意味, 文字 = 検索する("パソコンの電源が入らない", 問い合わせ集)

In [ ]:
# グラフで見る
y = np.arange(len(問い合わせ集))
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.barh(y - 0.2, 文字, 0.4, label="文字の一致(キーワード検索)", color="#A8AECC")
ax.barh(y + 0.2, 意味, 0.4, label="意味の近さ(埋め込み)", color="#3A46A8")
ax.set_yticks(y); ax.set_yticklabels(問い合わせ集); ax.invert_yaxis()
ax.set_xlabel("スコア"); ax.legend(fontsize=10); ax.grid(alpha=.3, axis="x")
ax.set_title("「パソコンの電源が入らない」で検索した結果")
plt.tight_layout(); plt.show()

print("キーワード検索は「社内Wi-Fiにつながらない」を1位にしました。")
print("「な」「い」「が」などの文字が共通しているだけで、意味とは関係ありません。")
print("意味検索は「PCが起動しません」を1位に。1文字も共通していないのに、です。")

> これが第11回の **RAG**(検索して答えさせる仕組み)の土台です。
> まず意味で関係のありそうな文書を探し、見つけた文書を大規模言語モデルに渡して答えを作らせます。

## 3. 話題ごとに集まる

スポーツ・料理・経済の3つの話題から4文ずつ、計12文を用意します。
**「これはスポーツです」というラベルは一切与えません。**それでも分かれるでしょうか。

In [ ]:
# ===== ③ 話題ごとに集まるか =====
話題 = {
    "スポーツ": ["昨日の試合は延長戦だった", "彼は今季15本目のホームランを打った",
                 "決勝でPK戦にもつれ込んだ", "監督が交代を告げた"],
    "料理":     ["玉ねぎをあめ色になるまで炒める", "小麦粉をふるってから加えます",
                 "弱火で20分ほど煮込みます", "塩を少々ふって味を調える"],
    "経済":     ["円相場が1ドル150円台に下落した", "日銀が政策金利を据え置いた",
                 "株価は3日続伸して取引を終えた", "消費者物価指数が前年比で上昇"],
}
文 = [s for v in 話題.values() for s in v]
ラベル = [k for k, v in 話題.items() for _ in v]
V = 数字にする(文)

同じ話題, 違う話題 = [], []
for i in range(len(文)):
    for j in range(i+1, len(文)):
        (同じ話題 if ラベル[i] == ラベル[j] else 違う話題).append(float(V[i] @ V[j]))

print(f"同じ話題どうしの平均: {np.mean(同じ話題):.3f}  ({len(同じ話題)}組)")
print(f"違う話題どうしの平均: {np.mean(違う話題):.3f}  ({len(違う話題)}組)")
print(f"差: {np.mean(同じ話題) - np.mean(違う話題):.3f}")
print()
print("「これはスポーツの文です」というラベルは一度も与えていません。")
print("文を入れて数字を出させ、近さを測っただけです。それでも話題ごとに分かれます。")

In [ ]:
# 384次元を2次元に押しつぶして、目で見る
from sklearn.decomposition import PCA
座標 = PCA(n_components=2).fit_transform(V)

plt.figure(figsize=(8.5, 5))
色 = {"スポーツ": "#3A46A8", "料理": "#E08A2B", "経済": "#02A96A"}
for k in 話題:
    印 = [i for i, l in enumerate(ラベル) if l == k]
    plt.scatter(座標[印, 0], 座標[印, 1], s=140, label=k, color=色[k], alpha=.85)
for i, s in enumerate(文):
    plt.annotate(s[:10], 座標[i], xytext=(6, 5), textcoords="offset points", fontsize=8.5)
plt.legend(); plt.grid(alpha=.3)
plt.title("似た文は、近くに集まる(384次元 → 2次元に圧縮)")
plt.tight_layout(); plt.show()

print("※ 2次元に押しつぶしているので、元の情報の一部しか見えていません。")
print("   離れて見えても実際は近い、ということもあります。目安として見てください。")

## 4. 教科書を疑う — 否定を見分けられるか

ここからが今日いちばん重要なところです。

埋め込みは「意味を理解する技術」と説明されることがあります。
本当にそうなら、**正反対の意味の文は遠くなる**はずです。確かめましょう。

In [ ]:
# ===== ④ 教科書を疑う:否定を見分けられるか =====
否定ペア = [
    ("この薬は安全です", "この薬は危険です"),
    ("この映画は面白い", "この映画は面白くない"),
    ("会議に出席します", "会議に出席しません"),
    ("値段が上がった", "値段が下がった"),
]
無関係ペア = [
    ("この映画は面白い", "経費精算の締切はいつですか"),
    ("会議に出席します", "今日は良い天気です"),
]

print("【意味が正反対のペア】")
for a, b in 否定ペア:
    print(f"  {近さ(a, b):.3f}  「{a}」×「{b}」")
print("\n【まったく無関係なペア(比較用)】")
for a, b in 無関係ペア:
    print(f"  {近さ(a, b):.3f}  「{a}」×「{b}」")

print()
print("正反対の意味の文のほうが、無関係な文よりずっと「近い」と判断されています。")
print("バグではありません。埋め込みは「一緒に出てくるか」を学んだので、")
print("同じ話題・同じ文型で使われる語は近くなります。設計どおりの結果です。")

In [ ]:
# 並べてグラフで見る
名前 = [f"{a[:8]}…\n× {b[:8]}…" for a, b in 否定ペア + 無関係ペア]
値   = [近さ(a, b) for a, b in 否定ペア + 無関係ペア]
色   = ["#E08A2B"] * len(否定ペア) + ["#A8AECC"] * len(無関係ペア)

plt.figure(figsize=(9, 4))
plt.bar(range(len(値)), 値, color=色)
plt.xticks(range(len(値)), 名前, fontsize=8.5)
plt.ylabel("近さ"); plt.ylim(0, 1)
plt.axhline(0.6, ls="--", c="crimson", lw=1)
plt.text(0.05, 0.62, "この線より上は「関連が強い」", color="crimson", fontsize=9.5)
plt.title("オレンジ=意味が正反対 / 灰色=まったく無関係")
plt.grid(alpha=.3, axis="y"); plt.tight_layout(); plt.show()

print("「この薬は安全です」×「この薬は危険です」が、赤い線を超えています。")
print("薬・契約・安全マニュアルなど、否定が意味を決める分野では致命的になりえます。")

> **結論**:埋め込みが測っているのは「意味」ではなく「**同じ場面で使われるか**」です。
> だから安全と危険が近くなります。この理解があれば、どこで失敗するかを予測できます。

## 5. 自分で試す

**壊しても大丈夫**です(開き直せば元に戻ります)。

In [ ]:
# ===== 自分で試す(課題用)=====
# (a) 自分で作った文のペアで測る
# ↓ この3組を、自分で考えた文に書きかえてください。
#    先に「いくつくらいになりそうか」を予想してから実行すること。
私のペア = [
    ("大学の講義は面白い", "授業がとても楽しい"),
    ("明日は雨が降るでしょう", "あすは晴れる見込みです"),
    ("この店は安い", "この店は高い"),
]
for a, b in 私のペア:
    print(f"  {近さ(a, b):.3f}  「{a}」×「{b}」")

# (b) 自分の文書を足して検索してみる
print("\n" + "=" * 55)
私の文書 = 問い合わせ集 + [
    "プリンタから紙が出てこない",
    "在宅勤務の申請はどうすればよいですか",
    "社員証を紛失しました",
]
検索する("印刷ができない", 私の文書)      # ← 質問を書きかえて試す

## 6. 提出課題(manaba)

**締切:次回講義の前日 23:59**

### 課題1:近さを手で計算する
次の4つの言葉が、3個の数で表されているとします。電卓を使ってかまいません。

```
東京   = (1,   0,   0)
大阪   = (0.8, 0.6, 0)
首都   = (0.6, 0.8, 0)
りんご = (0,   0,   1)
```

- (a) 6組すべての近さを計算しなさい
- (b) 2つのグループに分けなさい
- (c) 「首都」にいちばん近いのはどれですか。**それは事実として正しいですか。**
  おかしいと思ったら、その理由を書きなさい

### 課題2:ノートブックを動かして報告する
- (a) 自分で作った文のペア5組の「予想 / 実測」の表(**予想を先に書くこと**)
- (b) 否定・反対語のペア5組の表(無関係な文との比較つき)
- (c) 自分の分野(専攻・アルバイト・趣味)で、意味検索が役に立ちそうな場面を1つ、**具体的に**書く

### 課題3:次の主張を批判する(300字以上)

> 「社内規程を検索する AI を作りました。埋め込みで意味を理解するので、
> 言い方が違っても正しい規程が見つかります。これで問い合わせ対応の担当者は不要になります。」

考える材料:
- 「意味を理解する」は正確な言い方ですか。今日測った数字のどれが反証になりますか
- 規程で「〜してよい」と「〜してはならない」を取り違えると、何が起きますか
- 「正しい規程が見つかります」は、何件試して確かめた話ですか(第2回・第3回)
- 「見つけること」と「解釈して回答すること」は、同じ仕事ですか

**加点**:「担当者の仕事がどう変わるか」まで書けると評価が上がります。

---

採点は、正解かどうかよりも **試したことと考えたこと** を見ます。
課題2(a) は、予想が当たったかではなく、**外れたときになぜ外れたかを書けているか**を見ます。